In [1]:
import ase
from ase.build import graphene_nanoribbon
from ase.cluster import Icosahedron
import py3Dmol
import io
print("Building components...")

Building components...


In [2]:
# Create the Carbon Flake
flake = graphene_nanoribbon(4,4, type ='armchair',saturated=True,vacuum=5.0)
#Create the Iron Catalyst Cluster(Fe13)
fe13 = Icosahedron('Fe', noshells=2, latticeconstant=2.52)


In [3]:
#Create the FeCl3 Vapor Molecule
fecl3 =ase.Atoms('FeCl3',positions=[[0.0, 0.0, 0.0],       # Fe
                             [2.14, 0.0, 0.0],      # Cl 1
                             [-1.07, 1.853, 0.0],   # Cl 2
                             [-1.07, -1.853, 0.0]])#Cl3

In [4]:
flake.center(about=(0., 0., 0.))
fe13.center(about=(0., 0., 0.))
fecl3.center(about=(0., 0., 0.))

In [5]:
# Stack them vertically: 
# Place the Fe13 cluster 2.5 Angstroms above the carbon flake.
# Place the FeCl3 molecule 4.0 Angstroms above the Fe13 cluster.
fe13.positions[:, 2] += 2.5
fecl3.positions[:, 2] += (2.5 + 4.0)
combined_system = flake + fe13 + fecl3

In [6]:
# 6. Visualize with py3Dmol
xyz_file = io.StringIO()
ase.io.write(xyz_file, combined_system, format='xyz')
xyz_data = xyz_file.getvalue()
viewer = py3Dmol.view(width=800, height=500)
viewer.addModel(xyz_data, 'xyz')
viewer.setStyle({'sphere': {'scale': 0.25}, 'stick': {'radius': 0.1}})
viewer.zoomTo()
viewer.show()

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [7]:
import psi4

# Wipe all internal C++ timers and memory from previous runs
psi4.core.clean()

print("Initializing Psi4 for Optimization...")

# 1. Allocate computational resources
psi4.set_memory('6 GB')
psi4.set_num_threads(4) 

# 2. Tell Psi4 to write all the quantum mechanical math to a log file
psi4.core.set_output_file('FeCl3_optimization.dat', False)

# 3. Extract the coordinates of JUST the FeCl3 molecule
# Charge = 0, Multiplicity = 6
xyz_string = "0 6\n"
for atom in fecl3:
    xyz_string += f"{atom.symbol} {atom.position[0]} {atom.position[1]} {atom.position[2]}\n"

xyz_string += "symmetry c1\n"

# 4. Load the geometry into Psi4
molecule = psi4.geometry(xyz_string)

# 5. Set the theoretical models (DFT Functional and Basis Set)
psi4.set_options({
    'basis': 'def2-svp',   
    'reference': 'uhf',    
    'scf_type': 'df'       
})

# 6. Run the Geometry Optimization
print("Starting DFT structural relaxation. This may take a few minutes...")
final_energy = psi4.optimize('b3lyp')

print(f"\nOptimization Complete!")
print(f"Final Ground State Energy: {final_energy} Hartrees")

Some dependencies such as QCElemental have not yet finished migration to pydantic v2. If issues are encountered please downgrade pydantic or upgrade QCElemental as appropriate


Initializing Psi4 for Optimization...
Starting DFT structural relaxation. This may take a few minutes...
Optimizer: Optimization complete!

Optimization Complete!
Final Ground State Energy: -2643.9141477567123 Hartrees
